# Exercise — Choose a Governance Model for Trailhead Provisions (Solution)

**Trailhead Provisions** grew by acquisition: four domain teams — **E-commerce Orders** (led by the **E-commerce Orders Lead**), **Customer Identity & Loyalty** (led by the **Customer Identity Lead**), **Inventory & Fulfillment** (led by the **Inventory & Fulfillment Lead**), and **Marketing & Campaigns** (led by the **Marketing & Campaigns Lead**) — own their pipelines and definitions on a shared AWS platform. Alongside them sit three cross-cutting roles: a **Central Governance Lead**, a shared **Platform Engineering** team (manages pipeline infrastructure & OpenLineage instrumentation), and a Finance-appointed **CSRD Reporting Owner** (owns sustainability disclosure definitions).

A **GDPR audit** and a new **CSRD sustainability mandate** now create obligations that cross every domain. Decide how to govern. See `INSTRUCTIONS.md`.

## Step 0: Connect to your Cloud Lab
Run the cell below to configure your AWS Cloud Lab environment and seed the AWS Glue Data Catalog (`trailhead`) if it is not yet present in your lab account.

In [1]:
import os
os.environ["CARDINAL_LF_LIVE"] = "1"
os.environ["CARDINAL_AWS_REGION"] = "us-east-1"
# If your Vocareum AWS credentials are not already in the environment, paste them below:
# os.environ["AWS_ACCESS_KEY_ID"] = "..."
# os.environ["AWS_SECRET_ACCESS_KEY"] = "..."
# os.environ["AWS_SESSION_TOKEN"] = "..."

import boto3
import setup_aws
# Seed the Glue Data Catalog and S3 tables if not yet present in the lab account
glue = boto3.client("glue", region_name=os.environ["CARDINAL_AWS_REGION"])
try:
    glue.get_database(Name="trailhead")
except glue.exceptions.EntityNotFoundException:
    setup_aws.main()

created Glue database 'trailhead'
registered s3://123456789012-trailhead-data with Lake Formation
  customer               rows=  310  table-gaps=[]  untagged-cols=['email']
  orders                 rows= 1200  table-gaps=['owner']  untagged-cols=[]
  order_returns          rows=  106  table-gaps=[]  untagged-cols=[]
  supplier               rows=   40  table-gaps=['retention']  untagged-cols=[]
  product                rows=  200  table-gaps=['retention']  untagged-cols=['unit_cost']
  shipment               rows= 1109  table-gaps=['classification']  untagged-cols=[]
  campaign               rows=   20  table-gaps=['owner']  untagged-cols=[]
  campaign_membership    rows=  800  table-gaps=['owner', 'retention']  untagged-cols=['cust_ref']

Done. Glue database 'trailhead' seeded in s3://123456789012-trailhead-data with Lake Formation enforcement enabled.


In [2]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from governance_toolkit import make_catalog, lf_backend

gc = make_catalog()
print(f"Backend: {lf_backend()} -> {type(gc).__name__} (Glue DB: {gc.database}, Region: {gc.region})")
pd.DataFrame([{"domain": gc.catalog.table(t).domain, "table": t,
               "owner": gc.catalog.table(t).owner or "(no owner set)"}
              for t in gc.tables()]).sort_values("domain").reset_index(drop=True)

Backend: live -> AwsGovernedCatalog (Glue DB: trailhead, Region: us-east-1)


,domain,table,owner
0,Customer Identity & Loyalty,customer,customer-identity@trailhead.example
1,E-commerce Orders,orders,(no owner set)
2,E-commerce Orders,order_returns,orders-team@trailhead.example
3,Inventory & Fulfillment,supplier,inventory-fulfillment@trailhead.example
4,Inventory & Fulfillment,product,inventory-fulfillment@trailhead.example
5,Inventory & Fulfillment,shipment,inventory-fulfillment@trailhead.example
6,Marketing & Campaigns,campaign,(no owner set)
7,Marketing & Campaigns,campaign_membership,(no owner set)


## 1a. Governance-model decision
Choose **centralized**, **decentralized**, or **federated**, and state which decisions stay central and which side each role sits on.

### Decision — federated governance with a thin central core

Trailhead's domains own their data products and will resist a central team rewriting them, so
**centralized** governance stalls on friction. **Decentralized** is what they have today — and it
produced the audit failure (conflicting consent records, missing metadata, no lineage).
**Federated** keeps domain ownership while making a small set of platform-wide decisions
non-negotiable and central.

**Where each role sits in the operating model:**
- **Central / platform side:** the **Central Governance Lead** (enterprise data policy, classification taxonomy, consent standard, contract/SLO standards, and Lake Formation access baselines), the Finance-appointed **CSRD Reporting Owner** (authoritative carbon disclosure definition), and **Platform Engineering** (shared pipeline infrastructure and OpenLineage instrumentation).
- **Domain side:** the four domain leads — **E-commerce Orders Lead**, **Customer Identity Lead**, **Inventory & Fulfillment Lead**, and **Marketing & Campaigns Lead** — who own and operate their domain tables, pipelines, and per-product contracts.

**Decisions that stay CENTRAL:**
- **PII classification & tagging standard** — central (**Central Governance Lead**); GDPR exposure is platform-wide.
- **Consent system of record** — central (**Central Governance Lead** is Accountable for the enterprise consent standard, designating Loyalty's `customer.marketing_consent` maintained by the **Customer Identity Lead** as the single authoritative source); Loyalty and Marketing disagree today, creating direct GDPR liability.
- **Data contract standard & Data quality SLO policy** — central (**Central Governance Lead** sets and enforces the required contract schema and minimum SLO policy at the publish gate), while domain leads author the contracts and meet the SLOs on their products.
- **Shared metric definitions (per-order carbon)** — central (**CSRD Reporting Owner**); CSRD requires one auditable calculation.
- **Lineage instrumentation standard** — central/platform (**Platform Engineering**); cross-domain auditability requires consistent OpenLineage emission.

**Decisions that remain with the DOMAINS:**
- Schema evolution within a domain product, pipeline implementation, authoring domain contracts, and executing day-to-day quality remediation within the central guardrails.

## 1b. RACI (governance functions × domains)
One single individual **Accountable (`A`)** owner per function; name the enforcement authority and escalation path.

| Governance function | E-comm Orders Lead | Customer Identity Lead | Inventory & Fulfillment Lead | Marketing & Campaigns Lead | Accountable (single individual `A`) |
|---|---|---|---|---|---|
| PII classification & tagging | R | R | R | R | **Central Governance Lead** |
| Consent system of record | I | R | I | C | **Central Governance Lead** |
| Data contract standard | R | R | R | R | **Central Governance Lead** |
| Data quality SLO policy | R | R | R | R | **Central Governance Lead** |
| Lineage instrumentation | C | C | R | C | **Platform Engineering** |
| Carbon metric definition | I | I | R | C | **CSRD Reporting Owner** |
| Access policy (Lake Formation) | C | R | C | C | **Central Governance Lead** |

**Enforcement:** the **Central Governance Lead** holds publish-gate authority and can block any domain product from publishing if it fails the PII classification standard, the data contract schema, the minimum quality SLO policy, or the consent system-of-record rule.

**Escalation:** because consent and carbon cross domain boundaries, the **Central Governance Lead** (for privacy/consent/contracts) and **CSRD Reporting Owner** (for sustainability metrics) hold singular accountability (`A`). Unresolved cross-domain disputes (e.g., Marketing challenging Loyalty's consent feed) escalate to the **Data Governance Council**, chaired by the **Central Governance Lead** with the four domain leads and **CSRD Reporting Owner** as members.

## 1c. Governance topology
Draw the central core, the four domains, and the key cross-domain flows as a Mermaid diagram.

```mermaid
graph TD
    GC[Central Governance Core<br/>Central Governance Lead • CSRD Reporting Owner • Platform Engineering<br/>classification • consent SoR • contract & SLO standards • metrics]
    subgraph Domains
        D1[E-commerce Orders]
        D2[Customer Identity & Loyalty]
        D3[Inventory & Fulfillment]
        D4[Marketing & Campaigns]
    end
    GC -- standards & gates --> D1 & D2 & D3 & D4
    D2 -- loyalty consent --> GC -- authoritative consent SoR --> D4
    D1 -- order events --> D3
    D3 -- carbon source --> R[CSRD Quarterly Report]
    D1 -- order carbon --> R
```

---
## Solution Summary & Common Mistakes to Avoid

### Key points
- **Why not centralized:** four independently-shipping domains acquired from separate brands; a central team rewriting every pipeline stalls on organizational friction.
- **Why not decentralized:** that is today's state, and it produced the audit failure (conflicting consent records, missing catalog metadata, uninstrumented lineage).
- **Role alignment:** **Central Governance Lead**, **CSRD Reporting Owner**, and **Platform Engineering** sit on the central/platform side; the four domain leads (**E-commerce Orders Lead**, **Customer Identity Lead**, **Inventory & Fulfillment Lead**, **Marketing & Campaigns Lead**) sit on the domain side.
- **RACI with singular accountability:** every governance function has **one single individual `A`** (**Central Governance Lead** for classification, consent system of record, data contract standard, data quality SLO policy, and access policy; **Platform Engineering** for lineage instrumentation; **CSRD Reporting Owner** for carbon metric definition), while domain leads are `R` for executing on their domain data products.
- **Topology:** central core (`GC`) pushes standards & gates to all four domains; loyalty consent is governed through `GC` (`D2 -- loyalty consent --> GC -- authoritative consent SoR --> D4`); order and shipment carbon data flow from `D1` and `D3` into `R` (`CSRD Quarterly Report`).

### Common Mistakes to Avoid
- **Choosing centralized for "control"** without weighing the four-team political and delivery bottleneck cost.
- **A RACI with collective or multi-person `A`s** (such as `"Owning domain lead (per product)"`) — accountability must be singular (`A` = **Central Governance Lead** for the standard/policy, `R` = domain leads for per-product execution).
- **Making Customer Identity Lead `A` for cross-domain consent** while Marketing disputes it — because the consent conflict crosses domains, the **Central Governance Lead** must be `A` for the enterprise consent system-of-record standard, with **Customer Identity Lead** as `R`.
- **Listing roles in the RACI without placing them in the operating model (`1a`)** — explicitly state that **Platform Engineering** and **CSRD Reporting Owner** sit on the central/platform side.
- **A topology that omits the cross-domain flows** (consent routed through `GC`, and order → carbon → CSRD report) that drive the regulatory obligations.